# Daily Customer RFM Pipeline

This notebook runs the complete daily pipeline:

1. Load database credentials from environment variables.
2. Read non-cancelled orders from PostgreSQL.
3. Calculate recency, frequency, and monetary metrics.
4. Create 1–5 RFM scores and customer segments.
5. Prepare an idempotent upsert into `ecom.customer_rfm_daily`.


In [1]:
# Standard libraries
from datetime import date
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Custom modules
from src.db import get_connection, test_connection
from src.rfm import calculate_rfm_scores
from src.upsert import upsert_rfm_results

# Data analysis
import pandas as pd

In [2]:
# Connect using read credentials

read_conn = get_connection("read")

print(
    "Connection successful:",
    test_connection(read_conn)
)

Connection successful: True


In [8]:
# Load customer RFM metrics query

sql_file = (
    project_root
    / "sql"
    / "customer_rfm_metrics.sql"
)

with open(sql_file, "r") as file:
    query = file.read()

print("Query loaded successfully.")

Query loaded successfully.


In [10]:
read_conn.rollback()

print("Transaction reset")

Transaction reset


In [11]:
print(run_date)

2026-06-20


In [12]:
# Execute query and load data into dataframe

cursor = read_conn.cursor()

cursor.execute(
    query,
    {"run_date": run_date}
)

rows = cursor.fetchall()

columns = [desc[0] for desc in cursor.description]

rfm_df = pd.DataFrame(
    rows,
    columns=columns
)

print(rfm_df.shape)

rfm_df.head()

(8438, 5)


,customer_id,last_order_date,recency_days,frequency_orders,monetary_value
0,1,2026-05-25 07:57:11+00:00,26,7,88185.68
1,2,2026-05-07 05:03:47+00:00,44,1,3181.28
2,3,2026-05-25 08:00:08+00:00,26,1,9209.46
3,4,2026-05-27 10:19:04+00:00,24,4,28310.56
4,5,2026-04-12 16:53:45+00:00,69,1,10553.92


In [13]:
# Create RFM scores and customer segments

rfm_df = calculate_rfm_scores(
    rfm_df
)

rfm_df.head()

,customer_id,last_order_date,recency_days,frequency_orders,monetary_value,r_score,f_score,m_score,rfm_score,rfm_segment,run_date
0,1,2026-05-25 07:57:11+00:00,26,7,88185.68,4,5,5,455,Champions,2026-06-20
1,2,2026-05-07 05:03:47+00:00,44,1,3181.28,3,1,2,312,Others,2026-06-20
2,3,2026-05-25 08:00:08+00:00,26,1,9209.46,4,1,3,413,Others,2026-06-20
3,4,2026-05-27 10:19:04+00:00,24,4,28310.56,4,4,4,444,Champions,2026-06-20
4,5,2026-04-12 16:53:45+00:00,69,1,10553.92,2,1,3,213,Hibernating,2026-06-20


In [14]:
print(rfm_df.shape)

rfm_df.columns.tolist()

(8438, 11)


['customer_id',
 'last_order_date',
 'recency_days',
 'frequency_orders',
 'monetary_value',
 'r_score',
 'f_score',
 'm_score',
 'rfm_score',
 'rfm_segment',
 'run_date']

In [15]:
rfm_df["rfm_segment"].value_counts()

rfm_segment
Others          2060
Champions       1894
Hibernating     1731
At Risk         1344
Loyal            893
Big Spenders     516
Name: count, dtype: int64

In [16]:
# Create write connection

write_conn = get_connection(
    "write"
)

# Upsert results

upsert_rfm_results(
    write_conn,
    rfm_df
)

OperationalError: connection to server at "aws-0-ap-south-1.pooler.supabase.com" (3.108.251.216), port 5432 failed: FATAL:  (ENOIDENTIFIER) no tenant identifier provided (external_id or sni_hostname required)
connection to server at "aws-0-ap-south-1.pooler.supabase.com" (3.108.251.216), port 5432 failed: FATAL:  (ENOIDENTIFIER) no tenant identifier provided (external_id or sni_hostname required)


In [ ]:
read_conn.close()

if "write_conn" in locals():
    write_conn.close()

print(
    "Pipeline completed successfully."
)